In [1]:
import pandas as pd
import re
import os
import numpy as np
DB_OUTPUT = f"dataset\database_v3_imputed"
os.makedirs(DB_OUTPUT, exist_ok=True)

In [2]:
df = pd.read_csv(r"dataset\URA_renting_imputed_saits_no_cutoff_macro.csv")
# print(df['Project Name'].nunique())
print(len(df.columns.tolist()))
for col in df.columns:
    print(col)

171
project_idx
month_idx
date
is_post_cutoff
rent_psf_obs
rent_psf_imp
was_observed
condo_name
size_tier
size_label
district
segment
area_sqft_med
avg_psf_hist
n_transactions
Condo_Age_2026
tenure_remaining_years
tenure_freehold_like
tenure_medium_lease_60_80
tenure_more_than_80
tenure_short_lease_lt60
tenure_unknown
Large_Dev_200plus
project_size_small
project_size_medium
project_size_large
Planning Area
Neighbourhood
nearest_mrt_dist
nearest_mrt_name
nearest_bus_stops_dist
n_bus_stops_top3
nearest_supermarkets_dist
n_supermarkets_top3
nearest_parks_dist
n_parks_top3
nearest_clinics_dist
n_clinics_top3
nearest_bank_dist
n_bank_top3
nearest_atms_dist
n_atms_top3
nearest_school_dist
nearest_school_name
Nanyang Primary School
Rosyth School
Henry Park Primary School
Tao Nan School
Raffles Girls' Primary School
St. Hilda's Primary School
Pei Hwa Presbyterian Primary School
Methodist Girls' School (Primary)
Nan Hua Primary School
Chij St. Nicholas Girls' School
Anglo-Chinese School (Primar

In [3]:
df.head()

,project_idx,month_idx,date,is_post_cutoff,rent_psf_obs,rent_psf_imp,was_observed,condo_name,size_tier,size_label,...,nonlanded_private_units_in_the_pipeline_under_construction_share_lag1q,log1p_non_landed_units_launched_in_private_res_projects_with_prereq_for_sale_lag1q,non_landed_units_in_private_res_projects_with_prereq_not_launched_share_lag1q,log1p_unsold_completed_nonlanded_private_units_lag1q,log1p_unsold_non_landed_units_in_launched_private_res_projects_lag1q,assessed_non_landed_private_residential_owned_by_companies_share_lag1q,assessed_non_landed_private_residential_owned_by_pr_foreigners_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_companies_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_foreigners_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_spr_share_lag1q
0,0,0,1999-11,False,NaN,3.769841,False,1 MOULMEIN RISE,SZ3,medium,...,0.459061,9.601368,0.406074,NaN,7.834392,0.118301,0.150750,0.012868,0.088235,0.031250
1,0,1,1999-12,False,NaN,3.771444,False,1 MOULMEIN RISE,SZ3,medium,...,0.459061,9.601368,0.406074,NaN,7.834392,0.118301,0.150750,0.012868,0.088235,0.031250
2,0,2,2000-01,False,NaN,3.832254,False,1 MOULMEIN RISE,SZ3,medium,...,0.466325,9.481664,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812
3,0,3,2000-02,False,NaN,3.941147,False,1 MOULMEIN RISE,SZ3,medium,...,0.466325,9.481664,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812
4,0,4,2000-03,False,NaN,4.054842,False,1 MOULMEIN RISE,SZ3,medium,...,0.466325,9.481664,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812


In [4]:
# Rename columns and fix target column name
# Rename columns
df = df.rename(columns={
    "project_idx": "node_id",
    "month_idx": "timestep",
    "rent_psf_imp": "rent_per_sqft",
    "was_observed": "y_mask",
})

# Convert y_mask from True/False to 1/0
df["y_mask"] = df["y_mask"].astype(int)

# Duplicate columns for input features
df["node_id_feat"] = df["node_id"]
df["timestep_feat"] = df["timestep"]
df


C:\Users\cecel\AppData\Local\Temp\ipykernel_32932\2245120875.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["node_id_feat"] = df["node_id"]
C:\Users\cecel\AppData\Local\Temp\ipykernel_32932\2245120875.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["timestep_feat"] = df["timestep"]


,node_id,timestep,date,is_post_cutoff,rent_psf_obs,rent_per_sqft,y_mask,condo_name,size_tier,size_label,...,non_landed_units_in_private_res_projects_with_prereq_not_launched_share_lag1q,log1p_unsold_completed_nonlanded_private_units_lag1q,log1p_unsold_non_landed_units_in_launched_private_res_projects_lag1q,assessed_non_landed_private_residential_owned_by_companies_share_lag1q,assessed_non_landed_private_residential_owned_by_pr_foreigners_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_companies_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_foreigners_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_spr_share_lag1q,node_id_feat,timestep_feat
0,0,0,1999-11,False,NaN,3.769841,0,1 MOULMEIN RISE,SZ3,medium,...,0.406074,NaN,7.834392,0.118301,0.150750,0.012868,0.088235,0.031250,0,0
1,0,1,1999-12,False,NaN,3.771444,0,1 MOULMEIN RISE,SZ3,medium,...,0.406074,NaN,7.834392,0.118301,0.150750,0.012868,0.088235,0.031250,0,1
2,0,2,2000-01,False,NaN,3.832254,0,1 MOULMEIN RISE,SZ3,medium,...,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812,0,2
3,0,3,2000-02,False,NaN,3.941147,0,1 MOULMEIN RISE,SZ3,medium,...,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812,0,3
4,0,4,2000-03,False,NaN,4.054842,0,1 MOULMEIN RISE,SZ3,medium,...,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
711580,2258,310,2025-09,False,NaN,3.326864,0,ZENITH,SZ5,very_large,...,0.169813,5.010635,8.268475,0.049152,0.189677,0.000000,0.021667,0.125833,2258,310
711581,2258,311,2025-10,False,NaN,3.314321,0,ZENITH,SZ5,very_large,...,0.133148,5.170484,8.455743,0.047087,0.189534,0.000000,0.017421,0.093521,2258,311
711582,2258,312,2025-11,False,NaN,3.300587,0,ZENITH,SZ5,very_large,...,0.133148,5.170484,8.455743,0.047087,0.189534,0.000000,0.017421,0.093521,2258,312
711583,2258,313,2025-12,False,NaN,3.274982,0,ZENITH,SZ5,very_large,...,0.133148,5.170484,8.455743,0.047087,0.189534,0.000000,0.017421,0.093521,2258,313


In [15]:
# Drop unnecessary columns
df = df.drop(columns=["is_post_cutoff", "rent_psf_obs"])
df

,node_id,timestep,date,rent_per_sqft,y_mask,condo_name,size_tier,size_label,district,segment,...,non_landed_units_in_private_res_projects_with_prereq_not_launched_share_lag1q,log1p_unsold_completed_nonlanded_private_units_lag1q,log1p_unsold_non_landed_units_in_launched_private_res_projects_lag1q,assessed_non_landed_private_residential_owned_by_companies_share_lag1q,assessed_non_landed_private_residential_owned_by_pr_foreigners_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_companies_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_foreigners_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_spr_share_lag1q,node_id_feat,timestep_feat
0,0,0,1999-11,3.769841,0,1 MOULMEIN RISE,SZ3,medium,11,luxury,...,0.406074,NaN,7.834392,0.118301,0.150750,0.012868,0.088235,0.031250,0,0
1,0,1,1999-12,3.771444,0,1 MOULMEIN RISE,SZ3,medium,11,luxury,...,0.406074,NaN,7.834392,0.118301,0.150750,0.012868,0.088235,0.031250,0,1
2,0,2,2000-01,3.832254,0,1 MOULMEIN RISE,SZ3,medium,11,luxury,...,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812,0,2
3,0,3,2000-02,3.941147,0,1 MOULMEIN RISE,SZ3,medium,11,luxury,...,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812,0,3
4,0,4,2000-03,4.054842,0,1 MOULMEIN RISE,SZ3,medium,11,luxury,...,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
711580,2258,310,2025-09,3.326864,0,ZENITH,SZ5,very_large,10,luxury,...,0.169813,5.010635,8.268475,0.049152,0.189677,0.000000,0.021667,0.125833,2258,310
711581,2258,311,2025-10,3.314321,0,ZENITH,SZ5,very_large,10,luxury,...,0.133148,5.170484,8.455743,0.047087,0.189534,0.000000,0.017421,0.093521,2258,311
711582,2258,312,2025-11,3.300587,0,ZENITH,SZ5,very_large,10,luxury,...,0.133148,5.170484,8.455743,0.047087,0.189534,0.000000,0.017421,0.093521,2258,312
711583,2258,313,2025-12,3.274982,0,ZENITH,SZ5,very_large,10,luxury,...,0.133148,5.170484,8.455743,0.047087,0.189534,0.000000,0.017421,0.093521,2258,313


In [5]:
school_cols = [
    col for col in df.columns
    if "School" in col or "Institution" in col
]
dist_cols = [col for col in df.columns if "_dist" in col]
print("School columns:")
print(school_cols)
print(len(school_cols))
print("\nDistance columns:")
print(dist_cols)
print(len(dist_cols))

# Fill school_cols using the global max across all school_cols
global_school_max = np.ceil(df[school_cols].max().max())

df[school_cols] = df[school_cols].fillna(global_school_max)

# Fill dist_cols using each column's own max
for col in dist_cols:
    max_val = df[col].max()

    if pd.notna(max_val):
        df[col] = df[col].fillna(np.ceil(max_val))
df

School columns:
['Nanyang Primary School', 'Rosyth School', 'Henry Park Primary School', 'Tao Nan School', "Raffles Girls' Primary School", "St. Hilda's Primary School", 'Pei Hwa Presbyterian Primary School', "Methodist Girls' School (Primary)", 'Nan Hua Primary School', "Chij St. Nicholas Girls' School", 'Anglo-Chinese School (Primary)', 'Catholic High School (Primary)', 'Rulanh Primary School', 'Red Swastika School', 'Ai Tong School', "St. Joseph's Institution Junior", 'Kong Hwa School', 'South View Primary School', 'Chongfu School', 'Pei Chun Public School', "Holy Innocents' Primary School", 'Maris Stella High School (Primary)', "Singapore Chinese Girls' Primary School", 'Canberra Primary School', 'Radin Mas Primary School', 'River Valley Primary School', 'Gongshang Primary School', 'Temasek Primary School', 'Anderson Primary School', 'Princess Elizabeth Primary School']
30

Distance columns:
['nearest_mrt_dist', 'nearest_bus_stops_dist', 'nearest_supermarkets_dist', 'nearest_parks_

,node_id,timestep,date,is_post_cutoff,rent_psf_obs,rent_per_sqft,y_mask,condo_name,size_tier,size_label,...,non_landed_units_in_private_res_projects_with_prereq_not_launched_share_lag1q,log1p_unsold_completed_nonlanded_private_units_lag1q,log1p_unsold_non_landed_units_in_launched_private_res_projects_lag1q,assessed_non_landed_private_residential_owned_by_companies_share_lag1q,assessed_non_landed_private_residential_owned_by_pr_foreigners_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_companies_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_foreigners_share_lag1q,uncompleted_non_landed_private_residential_purchased_by_spr_share_lag1q,node_id_feat,timestep_feat
0,0,0,1999-11,False,NaN,3.769841,0,1 MOULMEIN RISE,SZ3,medium,...,0.406074,NaN,7.834392,0.118301,0.150750,0.012868,0.088235,0.031250,0,0
1,0,1,1999-12,False,NaN,3.771444,0,1 MOULMEIN RISE,SZ3,medium,...,0.406074,NaN,7.834392,0.118301,0.150750,0.012868,0.088235,0.031250,0,1
2,0,2,2000-01,False,NaN,3.832254,0,1 MOULMEIN RISE,SZ3,medium,...,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812,0,2
3,0,3,2000-02,False,NaN,3.941147,0,1 MOULMEIN RISE,SZ3,medium,...,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812,0,3
4,0,4,2000-03,False,NaN,4.054842,0,1 MOULMEIN RISE,SZ3,medium,...,0.447678,7.299121,7.770223,0.119037,0.148575,0.150972,0.041854,0.053812,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
711580,2258,310,2025-09,False,NaN,3.326864,0,ZENITH,SZ5,very_large,...,0.169813,5.010635,8.268475,0.049152,0.189677,0.000000,0.021667,0.125833,2258,310
711581,2258,311,2025-10,False,NaN,3.314321,0,ZENITH,SZ5,very_large,...,0.133148,5.170484,8.455743,0.047087,0.189534,0.000000,0.017421,0.093521,2258,311
711582,2258,312,2025-11,False,NaN,3.300587,0,ZENITH,SZ5,very_large,...,0.133148,5.170484,8.455743,0.047087,0.189534,0.000000,0.017421,0.093521,2258,312
711583,2258,313,2025-12,False,NaN,3.274982,0,ZENITH,SZ5,very_large,...,0.133148,5.170484,8.455743,0.047087,0.189534,0.000000,0.017421,0.093521,2258,313


In [6]:
# One-hot encode categorical columns
# Find non-numerical columns
non_numeric_cols = df.select_dtypes(exclude=['number']).columns.tolist()

print("Non-numerical columns:")
print(non_numeric_cols)

# Exclude these columns from one-hot encoding
exclude_cols = ["date", "condo_name"]

# Columns to one-hot encode
cols_to_encode = [
    col for col in non_numeric_cols
    if col not in exclude_cols
]

# One-hot encoding
df = pd.get_dummies(
    df,
    columns=cols_to_encode,
    dummy_na=False
)
df

Non-numerical columns:
['date', 'is_post_cutoff', 'condo_name', 'size_tier', 'size_label', 'segment', 'Planning Area', 'Neighbourhood', 'nearest_mrt_name', 'nearest_school_name']


,node_id,timestep,date,rent_psf_obs,rent_per_sqft,y_mask,condo_name,district,area_sqft_med,avg_psf_hist,...,nearest_school_name_Henry Park Primary School,nearest_school_name_Methodist Girls' School (Primary),nearest_school_name_Nanyang Primary School,nearest_school_name_Radin Mas Primary School,nearest_school_name_Raffles Girls' Primary School,nearest_school_name_River Valley Primary School,nearest_school_name_Singapore Chinese Girls' Primary School,nearest_school_name_St. Joseph's Institution Junior,nearest_school_name_Tao Nan School,nearest_school_name_Temasek Primary School
0,0,0,1999-11,NaN,3.769841,0,1 MOULMEIN RISE,11,1150.0,4.186846,...,False,False,False,False,False,False,False,True,False,False
1,0,1,1999-12,NaN,3.771444,0,1 MOULMEIN RISE,11,1150.0,4.186846,...,False,False,False,False,False,False,False,True,False,False
2,0,2,2000-01,NaN,3.832254,0,1 MOULMEIN RISE,11,1150.0,4.186846,...,False,False,False,False,False,False,False,True,False,False
3,0,3,2000-02,NaN,3.941147,0,1 MOULMEIN RISE,11,1150.0,4.186846,...,False,False,False,False,False,False,False,True,False,False
4,0,4,2000-03,NaN,4.054842,0,1 MOULMEIN RISE,11,1150.0,4.186846,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
711580,2258,310,2025-09,NaN,3.326864,0,ZENITH,10,2450.0,2.453515,...,False,False,False,False,False,True,False,False,False,False
711581,2258,311,2025-10,NaN,3.314321,0,ZENITH,10,2450.0,2.453515,...,False,False,False,False,False,True,False,False,False,False
711582,2258,312,2025-11,NaN,3.300587,0,ZENITH,10,2450.0,2.453515,...,False,False,False,False,False,True,False,False,False,False
711583,2258,313,2025-12,NaN,3.274982,0,ZENITH,10,2450.0,2.453515,...,False,False,False,False,False,True,False,False,False,False


In [ ]:
# Normalised all columns except target columns and fillna with 0
Normalising all columns Except date, rent_per_sqft, and condo_name

Non-numerical columns:
['date', 'condo_name', 'size_tier', 'size_label', 'segment', 'Planning Area', 'Neighbourhood', 'nearest_mrt_name', 'nearest_school_name']


,node_id,timestep,date,rent_per_sqft,y_mask,condo_name,district,area_sqft_med,avg_psf_hist,n_transactions,...,nearest_school_name_Henry Park Primary School,nearest_school_name_Methodist Girls' School (Primary),nearest_school_name_Nanyang Primary School,nearest_school_name_Radin Mas Primary School,nearest_school_name_Raffles Girls' Primary School,nearest_school_name_River Valley Primary School,nearest_school_name_Singapore Chinese Girls' Primary School,nearest_school_name_St. Joseph's Institution Junior,nearest_school_name_Tao Nan School,nearest_school_name_Temasek Primary School
0,0,0,1999-11,3.769841,0,1 MOULMEIN RISE,11,1150.0,4.186846,149,...,False,False,False,False,False,False,False,True,False,False
1,0,1,1999-12,3.771444,0,1 MOULMEIN RISE,11,1150.0,4.186846,149,...,False,False,False,False,False,False,False,True,False,False
2,0,2,2000-01,3.832254,0,1 MOULMEIN RISE,11,1150.0,4.186846,149,...,False,False,False,False,False,False,False,True,False,False
3,0,3,2000-02,3.941147,0,1 MOULMEIN RISE,11,1150.0,4.186846,149,...,False,False,False,False,False,False,False,True,False,False
4,0,4,2000-03,4.054842,0,1 MOULMEIN RISE,11,1150.0,4.186846,149,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
711580,2258,310,2025-09,3.326864,0,ZENITH,10,2450.0,2.453515,9,...,False,False,False,False,False,True,False,False,False,False
711581,2258,311,2025-10,3.314321,0,ZENITH,10,2450.0,2.453515,9,...,False,False,False,False,False,True,False,False,False,False
711582,2258,312,2025-11,3.300587,0,ZENITH,10,2450.0,2.453515,9,...,False,False,False,False,False,True,False,False,False,False
711583,2258,313,2025-12,3.274982,0,ZENITH,10,2450.0,2.453515,9,...,False,False,False,False,False,True,False,False,False,False


School columns:
["Methodist Girls' School (Primary)", 'Anglo-Chinese School (Primary)', 'Catholic High School (Primary)', "St. Joseph's Institution Junior", 'Maris Stella High School (Primary)', 'nearest_school_name_Anglo-Chinese School (Primary)', 'nearest_school_name_Catholic High School (Primary)', "nearest_school_name_Methodist Girls' School (Primary)", "nearest_school_name_St. Joseph's Institution Junior"]

Distance columns:
['nearest_mrt_dist', 'nearest_bus_stops_dist', 'nearest_supermarkets_dist', 'nearest_parks_dist', 'nearest_clinics_dist', 'nearest_bank_dist', 'nearest_atms_dist', 'nearest_school_dist']


In [ ]:
# Fillna for distance columns with ceiling of the max value in each column

In [ ]:
# Normalised all columns except target columns and fillna with 0

In [6]:
# Print columns that are NOT numerical
non_numeric_cols = df.select_dtypes(exclude=['number']).columns.tolist()

print("Non-numerical columns:")
print(non_numeric_cols)

Non-numerical columns:
['date', 'is_post_cutoff', 'was_observed', 'condo_name', 'size_tier', 'size_label', 'segment', 'Planning Area', 'Neighbourhood', 'nearest_mrt_name', 'nearest_school_name']


In [ ]:
nodes_output = os.path.join(DB_OUTPUT, "nodes")
timestep_output = os.path.join(DB_OUTPUT, "timestep")

os.makedirs(nodes_output, exist_ok=True)
os.makedirs(timestep_output, exist_ok=True)

# Split by project_idx. Example: project_idx 1 -> 0001.csv
for project_idx, project_df in df.groupby("project_idx"):
    filename = f"{int(project_idx):04d}.csv"
    output_path = os.path.join(nodes_output, filename)
    project_df.sort_values("month_idx").to_csv(output_path, index=False)

# Split by timestep. Example: month_idx 0 and date 1999-11 -> 001-199911.csv
for (month_idx, date), timestep_df in df.groupby(["month_idx", "date"]):s
    date_label = pd.to_datetime(date).strftime("%Y%m")
    filename = f"{int(month_idx):03d}-{date_label}.csv"
    output_path = os.path.join(timestep_output, filename)
    timestep_df.sort_values("project_idx").to_csv(output_path, index=False)

print(f"Wrote {df['project_idx'].nunique()} node files to {nodes_output}")
print(f"Wrote {df[['month_idx', 'date']].drop_duplicates().shape[0]} timestep files to {timestep_output}")

Wrote 2259 node files to dataset\database_v3_imputed\nodes
Wrote 315 timestep files to dataset\database_v3_imputed\timestep
